## Tarea 4
Programa Experto en Inteligencia Artificial con Python  
ML3006 - Web Mining en Python  
Estudiante: Natalia Bonilla Villalobos.  
**XPath y LLMs para Web Scraping**

<a id="menu"></a>

# Menú
- [Ejercicio 1](#ej1)
- [Ejercicio 2](#ej2)

In [1]:
import numpy as np
import requests
import json
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from io import StringIO
from lxml import html

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from urllib.parse import urljoin

<a id="ej1"></a>
# Ejercicio 1

[50 puntos] Para este ejercicio utilizaremos la categoría de frijoles del sitio Maxi Pali disponible en este [enlace](https://www.maxipali.co.cr/abarrotes/arroz-frijol-y-semillas/frijol). Usando expresiones `XPath` y `requests + lxml` construya un script que:
- a) Descargue todas las tarjetas de producto presentes en la página.
- b) Extraiga para cada producto: nombre/presentación, código de producto, precio actual, precio anterior (si existe) y porcentaje de descuento.
- c) Estructure los datos en un `DataFrame` de `pandas` y guárdelos en un archivo `frijoles_maxipali.csv`

[↑ Volver al Menú](#menu)

### Navegar y extraer datos

In [2]:
LISTING_URL = 'https://www.maxipali.co.cr/abarrotes/arroz-frijol-y-semillas/frijol'
BASE_URL = "https://www.maxipali.co.cr"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

## Parte 1
- a) Descargue todas las tarjetas de producto presentes en la página.

In [3]:
listing_response = requests.get(LISTING_URL, headers=HEADERS, timeout=30)
listing_response.raise_for_status()

listing_dom = html.fromstring(listing_response.content)
# vtex-search-result-3-x-galleryItem -> cada card
product_cards = listing_dom.xpath("//div[contains(@class,'vtex-search-result-3-x-galleryItem')]")

print(f"Respuesta HTTP {listing_response.status_code} · {len(listing_response.content)} bytes descargados")
print(f"Tarjetas encontradas: {len(product_cards)}")

Respuesta HTTP 200 · 1573489 bytes descargados
Tarjetas encontradas: 10


In [4]:
item1 = listing_dom.xpath("(//div[contains(@class,'vtex-search-result-3-x-galleryItem')])[1]")[0]

# para limpiar el texto de un elemento, evitando errores por elementos no encontrados.
def first_text(element, xpath):
    res = element.xpath(xpath)
    return res[0].strip() if res else None


nombre = first_text(item1, ".//span[@id='product-summary-sku-name']/text()[normalize-space()]")
precio_actual = first_text(item1, ".//div[contains(@class,'sellingPrice')]//span//text()[contains(.,'₡')]")
precio_anterior = first_text(item1, ".//div[contains(@class,'listPrice')]//span//text()[contains(.,'₡')]")
descuento = first_text(item1, ".//span[contains(@class,'vtex-product-price-1-x-savingsPercentage')]/text()")

# print('nombre:', item1.text_content().strip())
print('nombre:', nombre)
print('precio_anterior:', precio_actual)
print('precio_anterior:', precio_actual)
print('descuento:', descuento)

nombre: Frijol negro indiana primera calidad empacados - 800 g
precio_anterior: ₡1.000
precio_anterior: ₡1.000
descuento: None


## Parte 2
- b) Extraiga para cada producto: nombre/presentación, código de producto, precio actual, precio anterior (si existe) y porcentaje de descuento.

In [ ]:
# entra en la url del producto y obtiene el codigo de barras
# el codigo de barras no se encuentra en el listado, por lo que hay que entrar a la url del producto para obtenerlo.
def obtener_cod(url):
    if not url:
        return None
    
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    doc = html.fromstring(response.content)
    
    # return first_text(doc, "//span[.//text()='EAN]//span[contains(@class,'product-identifier__value')]/text()]")
    return first_text(doc, "//span[contains(@class,'product-identifier__value')]/text()[normalize-space()]")

In [ ]:
# "precio_actual": first_text(doc,
#     "(//div[contains(@class,'priceContainer') and not(ancestor::div[@id='priceFooterFixed'])]"
#     "//div[contains(@class,'sellingPrice')]//span[text()[contains(.,'₡')]])[1]/text()"),
# "precio_anterior": first_text(doc,
#     "(//div[contains(@class,'priceContainer') and not(ancestor::div[@id='priceFooterFixed'])]"
#     "//div[contains(@class,'listPrice')]//span[text()[contains(.,'₡')]])[1]/text()"),
# "descuento": first_text(doc,
#     "(//div[contains(@class,'priceContainer') and not(ancestor::div[@id='priceFooterFixed'])]"
#     "//span[contains(@class,'savingsPercentage')])[1]/text()")

In [ ]:
# entra en la url del producto y obtiene los precio sy descuento
# la idea de intentarlo asi era para revisar que nos trajeramos los precios reales.
def obtener_precios(url):
    if not url:
        return None
    
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    doc = html.fromstring(response.content)
    
    return {
        "precio_actual": first_text(doc, ".//div[contains(@class,'sellingPrice')]//span//text()[contains(.,'₡')]"),
        "precio_anterior": first_text(doc, ".//div[contains(@class,'listPrice')]//span//text()[contains(.,'₡')]"),
        "descuento": first_text(doc, ".//span[contains(@class,'vtex-product-price-1-x-savingsPercentage')]/text()")
    }

In [39]:
tarjetas = []
for producto in product_cards:
    # url para el detalle del producto
    href = first_text(producto, ".//a[contains(@href,'/p')]/@href")
    url_detalle = f"https://www.maxipali.co.cr{href}" if href else None
    
    # codigo del producto, codigo de barras
    codigo_producto = obtener_cod(url_detalle) if url_detalle else None
    
    # precios y descuento
    precios = obtener_precios(url_detalle) if url_detalle else {}
    
    # nombre del producto
    nombre_producto = first_text(producto, ".//span[@id='product-summary-sku-name']/text()[normalize-space()]")
    
    tarjetas.append({
        'codigo_producto': codigo_producto,
        'nombre_producto': nombre_producto,
        'precio_actual': precios.get("precio_actual"),
        'precio_anterior': precios.get("precio_anterior"),
        'descuento': precios.get("descuento")
    })

## Parte 3
- c) Estructure los datos en un `DataFrame` de `pandas` y guárdelos en un archivo `frijoles_maxipali.csv`

In [42]:
df = pd.DataFrame(tarjetas)
df

,codigo_producto,nombre_producto,precio_actual,precio_anterior,descuento
0,7441078256679,Frijol negro indiana primera calidad empacados...,₡1.000,None,None
1,7441078256709,Frijol rojo Indiana bolsa - 800 g,₡1.100,None,None
2,7441078258031,Frijol Suli Rojo - 700 g,₡690,None,None
3,7441078258048,Frijol Suli negro buena fuente de fibra segund...,₡680,None,None
4,7441074152203,Frijol Blanco Indiana -400gr,₡620,None,None
5,7441078260515,Frijol Suli Rojo - 800 g,₡900,None,None
6,7441078256686,Frijol Rojo Indiana Premium - 800 g,₡1.200,None,None
7,7441078260522,Frijol Suli Negro Primera Calidad Empacados - ...,₡850,None,None
8,7441074152692,Frijol Rojo Indiana -3000gr,₡4.200,None,None
9,7441078257355,Frijol Suli Rojo - 3 kg,₡3.500,None,None


In [41]:
df.to_csv("frijoles_maxipali.csv", index=False)

<a id="ej2"></a>
# Ejercicio 2

[50 puntos] Desde la categoría de alquileres de Encuentra24 disponible en este [enlace](https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/costa-rica-es-san-jose-provincia-montes-de-oca), extraiga la información de los avisos publicados. Puede elegir una de las dos opciones siguientes:   

**Opción A: con `chatlas` (asistido por LLM):**
- a) Recorra las primeras 3 páginas de la paginación y recupere la descripción de cada aviso.
- b) Defina un esquema de datos con `pydantic` que incluya al menos: alquiler, recámaras, baños, metros cuadrados construidos, ubicación, fecha de publicación, renta por metro
cuadrado, alquiler por metro de terreno, metros totales, disponibilidad, dirección, costos de mantenimiento, altura, años de construcción, niveles, piscina, tipo de piso, n´umero
de piso, parqueo, línea blanca, balcón o terraza, detalles ampliados, cuota condominal y amenidades.
- c) Procese cada aviso con `chatlas` y almacene la salida en `alquileres_encuentra24.csv`.   

**Opción B: solo con `XPath / lxml` (sin LLM:)**
- a) Recorra las primeras 3 páginas de la paginación y descargue el HTML de cada aviso.
- b) Usando expresiones XPath extraiga de cada aviso al menos: alquiler, recámaras, baños, metros cuadrados construidos, ubicación, fecha de publicación, disponibilidad, dirección, parqueo, amenidades y cualquier otro campo disponible en la ficha.
- c) Limpie y normalice los valores extraídos (eliminar símbolos de moneda, convertir a tipos num´ericos, etc.).
- d) Estructure los datos en un DataFrame de pandas y guárdelos en `alquileres_encuentra24.csv`.


**NOTA**: documente claramente su flujo de trabajo (captura de HTML, limpieza, envío al modelo) y comente cualquier limitación o error relevante encontrado durante el scraping.  

**ENTREGABLES**: Junto con el notebook de Python (Tarea4_nombre_apellido.ipynb), debe entregar también los archivos `CSV` generados: `frijoles_maxipali.csv` y `alquileres_encuentra24.csv`



[↑ Volver al Menú](#menu)

**Opción B:**

## Parte 1
- a) Recorra las primeras 3 páginas de la paginación y descargue el HTML de cada aviso.

In [48]:
# URL = "https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/san-pedro-montes-de-oca-internet-simetrico-30-mb-fibra-optica/31049785"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Connection": "keep-alive",
}

URLS = [
    "https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/san-jose-provincia-montes-de-oca",
    "https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos.2/san-jose-provincia-montes-de-oca",
    "https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos.3/san-jose-provincia-montes-de-oca",
]

In [49]:
for i, url in enumerate(URLS, start=1):
    response = requests.get(url, headers=HEADERS, timeout=10)
    
    # Guardamos cada pagina en su propio archivo
    with open(f"pagina{i}.html", "w", encoding="utf-8") as f:
        f.write(response.text)
    
    print(f"Página {i} - Status: {response.status_code}")
    
    # Esperamos 30 segundos entre cada descarga
    time.sleep(30)

print(f"Respuesta HTTP {response.status_code} · {len(response.content)} bytes descargados")

Página 1 - Status: 200
Página 2 - Status: 200
Página 3 - Status: 200
Respuesta HTTP 200 · 719675 bytes descargados


## Parte 2
- b) Usando expresiones XPath extraiga de cada aviso al menos: `alquiler`, `recámaras`, `baños`, `metros cuadrados construidos`, `ubicación`, `fecha de publicación`, `disponibilidad`, `dirección`, `parqueo`, `amenidades` y cualquier otro campo disponible en la ficha.
- c) Limpie y normalice los valores extraídos (eliminar símbolos de moneda, convertir a tipos num´ericos, etc.).

In [50]:
with open("pagina1.html", "r", encoding="utf-8") as f:
    contenido = f.read()

# Creamos el arbol DOM
arbol = html.fromstring(contenido)

# Buscamos todos los cards por su clase
cards = arbol.xpath('//a[contains(@class, "item-card-link")]')

print(f"Cards encontrados: {len(cards)}")

Cards encontrados: 20


In [128]:
def first_text2(element, xpath):
    res = element.xpath(xpath)
    
    if not res:
        return None
    
    # Si son strings
    if isinstance(res[0], str):
        text = ' '.join(r.strip() for r in res if isinstance(r, str))
        return text.strip()
    
    # Si son nodos
    text = ' '.join(r.text_content().strip() for r in res)
    return text.strip()

In [129]:
primer_card = cards[0]
precio = primer_card.xpath('.//span[contains(@class, "card_price")]//text()')
print("Precio:", precio)

# Ubicación
ubicacion = first_text2(primer_card, './/p[contains(@class, "card_subtitle")]//text()')
print("Ubicación:", ubicacion)

# Título
titulo = first_text2(primer_card, './/h3[contains(@class, "card_title")]//text()')
print("Título:", titulo)

# Especificaciones (recámaras, baños, m²)
specs = primer_card.xpath('.//span[contains(@class, "card_spec")]//text()')
print("Specs:", specs)

# URL del aviso
enlace = first_text2(primer_card, './@href')
url_detalle = f"https://www.encuentra24.com{enlace}" if enlace else None
print("URL:", url_detalle)

# Descripción
descripcion = first_text2(primer_card, './/p[contains(@class, "card_description")]//text()')
print("Descripción:", descripcion)

Precio: [' ', '₡', ' ', '625,000', ' ', '₡', ' ', '625,000']
Ubicación: San Pedro, Montes de Oca
Título: Alquiler Hermoso Apartamento En San Pedro 3 Habitaciones - Cerca De La Ucr
Specs: ['3 Recámaras', '1 Baños', '130 m²']
URL: https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/alquiler-hermoso-apartamento-en-san-pedro-3-habitaciones-cerca-de-la-ucr/32227028
Descripción: Disponibilidad inmediata - Amplia sala y comedor Ubicado en zona de San Pedro Vargas Araya, a pasos de la UCR, esta propiedad en segundo nivel con enfoque en finos acabados; destacan sus maderas de genízaro y almendro que dan ese detallado único. Descripción: - Habitación principal con walk-in closet y balcón - Habitación secundaria con closet integrado - Un baño amplio completo - tercera habitación con closet - Sala y comedor - Balcón - Cocina con extractor y muebles - Área de lavado interno - 1 Estacionamiento con portón eléctrico Alquiler incluye mantenimiento de la propiedad. Esta es zon

In [139]:
def otros_detalles(url):
    if not url:
        return None
    
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    doc = html.fromstring(response.content)
    
    # fecha = first_text(doc, '//div[contains(@class,"flex items-center gap-2 text-xs text-muted-foreground")]//text()')
    # fecha = doc.xpath('//div[contains(@class,"flex items-center gap-2 text-xs text-muted-foreground")]//text()')
    fecha = first_text2(doc, '//span[contains(text(),"Actualizado el")]/text()')
    if fecha:
        fecha = fecha.replace('Actualizado el:', '').strip()
    # amenidades = doc.xpath('//div[contains(@class,"grid")]//span[@class="text-sm text-foreground"]/text()')
    
    amenidades = doc.xpath('//h2[contains(text(),"Amenidades y servicios")]'
                           '/following::div[contains(@class,"grid")][1]'
                           '//span[@class="text-sm text-foreground"]/text()')
    
    # quitar duplicados
    amenidades = [a.strip() for a in amenidades if a.strip()]
    amenidades = list(dict.fromkeys(amenidades))  # mantiene orden    
      
    parqueo = any("Parking" in a or "Garaje" in a for a in amenidades)
        
    detalles_keys = doc.xpath('//h2[contains(text(),"Detalles adicionales")]'
                              '/following::p[@class="text-xs text-muted-foreground whitespace-nowrap"]/text()')

    detalles_vals = doc.xpath('//h2[contains(text(),"Detalles adicionales")]'
                              '/following::p[@class="text-sm font-medium text-foreground"]/text()')

    detalles_dict = dict(zip(detalles_keys, detalles_vals))    
    
    return {
        'fecha': fecha,
        'amenidades': amenidades,
        'parqueo': parqueo,
        'detalles': detalles_dict
    }

# first_text(doc, "//span[contains(@class,'product-identifier__value')]/text()[normalize-space()]")

otros_detalles(url_detalle)

{'fecha': '17/04/2026',
 'amenidades': ['Cerca de Escuela',
  'Cerca del Tráfico',
  'Vista a las Montañas',
  'Seguridad 24 Horas',
  'Lavandería interna',
  'Desayunador'],
 'parqueo': False,
 'detalles': {'Número de Piso': '2',
  'Pet friendly': 'Sí',
  'Tipo de propiedad': 'Apartamento',
  'Niveles de la propiedad': '1',
  'Área total': '150 m²',
  'Área construida': '130 m²'}}

In [ ]:
def extraer_card(card):
    precio      = card.xpath('.//span[contains(@class, "card_price")]//text()')
    ubicacion   = card.xpath('.//p[contains(@class, "card_subtitle")]//text()')
    titulo      = card.xpath('.//h3[contains(@class, "card_title")]//text()')
    specs       = card.xpath('.//span[contains(@class, "card_spec")]//text()')
    url         = card.xpath('./@href')
    descripcion = card.xpath('.//p[contains(@class, "card_description")]//text()')

    url_detalle = "https://www.encuentra24.com" + url[0] if url else ""

    # precio_texto = ''.join(precio)
    # precio_texto = re.findall(r'\d+', precio_texto)
    # precio_num = int(precio_texto[0]) if precio_texto else None
    numeros = re.findall(r'\d[\d,]*', ''.join(precio))

    if numeros:
        precio_num = int(numeros[0].replace(',', ''))
    else:
        precio_num = None
    
    titulo = ' '.join(titulo).strip() if titulo else ""
    ubicacion = ' '.join(ubicacion).strip() if ubicacion else ""
    descripcion = ' '.join(descripcion).strip() if descripcion else ""

    recamaras = None
    banos = None
    metros = None

    for s in specs:
        s = s.strip()
        
        if 'Recámaras' in s:
            recamaras = int(s.split()[0])
        elif 'Baños' in s:
            banos = float(s.split()[0])
        elif 'm²' in s:
            metros = int(s.split()[0])

    detalles = otros_detalles(url_detalle) if url_detalle else {}
    if not detalles:
        detalles = {}

    return {
        "titulo": titulo,
        "precio": precio_num,
        "ubicacion": ubicacion,
        "recamaras": recamaras,
        "banos": banos,
        "m2": metros,
        "url": url_detalle,
        "descripcion": descripcion,
        **detalles
    }

In [180]:
# Primer card
with open("pagina1.html", "r", encoding="utf-8") as f:
    contenido = f.read()

arbol = html.fromstring(contenido)
cards = arbol.xpath('//a[contains(@class, "item-card-link")]')

resultado = extraer_card(cards[0])

for campo, valor in resultado.items():
    print(f"{campo}: {valor}")

titulo: Alquiler Hermoso Apartamento En San Pedro 3 Habitaciones - Cerca De La Ucr
precio: 625000
ubicacion: San Pedro, Montes de Oca
recamaras: 3
banos: 1.0
m2: 130
url: https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/alquiler-hermoso-apartamento-en-san-pedro-3-habitaciones-cerca-de-la-ucr/32227028
fecha: 17/04/2026
amenidades: ['Cerca de Escuela', 'Cerca del Tráfico', 'Vista a las Montañas', 'Seguridad 24 Horas', 'Lavandería interna', 'Desayunador']
parqueo: False
detalles: {'Número de Piso': '2', 'Pet friendly': 'Sí', 'Tipo de propiedad': 'Apartamento', 'Niveles de la propiedad': '1', 'Área total': '150 m²', 'Área construida': '130 m²'}


In [175]:
print(extraer_card(cards[0]))

{'titulo': 'Alquiler Hermoso Apartamento En San Pedro 3 Habitaciones - Cerca De La Ucr', 'precio': 625000625000, 'ubicacion': 'San Pedro, Montes de Oca', 'recamaras': 3, 'banos': 1.0, 'm2': 130, 'url': 'https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/alquiler-hermoso-apartamento-en-san-pedro-3-habitaciones-cerca-de-la-ucr/32227028', 'descripcion': 'Disponibilidad inmediata - Amplia sala y comedor Ubicado en zona de San Pedro Vargas Araya, a pasos de la UCR, esta propiedad en segundo nivel con enfoque en finos acabados; destacan sus maderas de genízaro y almendro que dan ese detallado único. Descripción: - Habitación principal con walk-in closet y balcón - Habitación secundaria con closet integrado - Un baño amplio completo - tercera habitación con closet - Sala y comedor - Balcón - Cocina con extractor y muebles - Área de lavado interno - 1 Estacionamiento con portón eléctrico Alquiler incluye mantenimiento de la propiedad. Esta es zona residencial por lo 

In [183]:
paginas = ['pagina1.html', 'pagina2.html', 'pagina3.html']
todos_los_datos = []

for pagina in paginas:
    with open(pagina, "r", encoding="utf-8") as f:
        contenido = f.read()
    
    arbol = html.fromstring(contenido)
    cards = arbol.xpath('//a[contains(@class, "item-card-link")]')
    
    for card in cards:
        resultado = extraer_card(card)
        todos_los_datos.append(resultado)

# Verificamos algunos resultados
for item in todos_los_datos[:3]:
    print(item)

{'titulo': 'Alquiler Hermoso Apartamento En San Pedro 3 Habitaciones - Cerca De La Ucr', 'precio': 625000, 'ubicacion': 'San Pedro, Montes de Oca', 'recamaras': 3, 'banos': 1.0, 'm2': 130, 'url': 'https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/alquiler-hermoso-apartamento-en-san-pedro-3-habitaciones-cerca-de-la-ucr/32227028', 'fecha': '17/04/2026', 'amenidades': ['Cerca de Escuela', 'Cerca del Tráfico', 'Vista a las Montañas', 'Seguridad 24 Horas', 'Lavandería interna', 'Desayunador'], 'parqueo': False, 'detalles': {'Número de Piso': '2', 'Pet friendly': 'Sí', 'Tipo de propiedad': 'Apartamento', 'Niveles de la propiedad': '1', 'Área total': '150 m²', 'Área construida': '130 m²'}}
{'titulo': 'Alquiler De Apto. Estudio, Incluye Servicios. San Pedro.', 'precio': 280000, 'ubicacion': 'Montes de Oca, San José provincia', 'recamaras': None, 'banos': 1.0, 'm2': None, 'url': 'https://www.encuentra24.com/costa-rica-es/bienes-raices-alquiler-apartamentos/alquiler-d

## Parte 3
- d) Estructure los datos en un DataFrame de pandas y guárdelos en `alquileres_encuentra24.csv`.

In [213]:
df = pd.DataFrame(todos_los_datos)
df.sample(6)

,titulo,precio,ubicacion,recamaras,banos,m2,url,fecha,amenidades,parqueo,detalles
43,"Alquiler De Apartamento Estudio, Gimnasio, Cow...",280000,"San Pedro, Montes de Oca",NaN,1.0,20.0,https://www.encuentra24.com/costa-rica-es/bien...,05/04/2026,"[Seguridad 24 Horas, Lavandería interna, Gimna...",False,"{'Número de Piso': '2', 'Pet friendly': 'Sí', ..."
54,Apartamento En Alquiler Amoblado Torre Los Yos...,1400,"San Pedro, Montes de Oca",2.0,1.0,62.0,https://www.encuentra24.com/costa-rica-es/bien...,16/04/2026,"[Cerca de Escuela, Cerca del Tráfico, Vista al...",True,"{'Piscina': 'Sí', 'Número de Piso': '14', 'Pet..."
28,Alquiler De Habitación Amoblada En Edificio Co...,250000,"San Pedro, Montes de Oca",NaN,1.0,20.0,https://www.encuentra24.com/costa-rica-es/bien...,12/04/2026,"[Cerca de Escuela, Amoblado, Nevera, Microonda...",False,"{'Número de Piso': 'Apartamento', 'Pet friendl..."
8,Alquiler De Apartamento En Montes De Oca-san P...,1100,"San Pedro, Montes de Oca",2.0,1.0,87.0,https://www.encuentra24.com/costa-rica-es/bien...,16/04/2026,"[Cerca del Tráfico, Parking bajo techo, Seguri...",True,"{'Piscina': 'Sí', 'Tipo de propiedad': 'Aparta..."
4,Apartamento Estudio Con Servicios,215000,"San Pedro, Montes de Oca",NaN,1.0,NaN,https://www.encuentra24.com/costa-rica-es/bien...,12/04/2026,[],False,"{'Tipo de propiedad': 'Apartamento', 'Balcón/T..."
55,Alquiler Penthouse De Lujo En Latitud Los Yoses,2600,"San Pedro, Montes de Oca",4.0,3.0,NaN,https://www.encuentra24.com/costa-rica-es/bien...,16/04/2026,"[Cerca de Escuela, Vista a las Montañas, Parki...",True,"{'Piscina': 'Sí', 'Número de Piso': '22', 'Tip..."


In [214]:
df.shape

(60, 11)

In [215]:
# Expandir lista a filas
amenidades_exp = df['amenidades'].explode()

# Crear columnas binarias
amenidades_dummies = pd.get_dummies(amenidades_exp, dtype=int)

# Volver a agrupar por fila original
amenidades_dummies = amenidades_dummies.groupby(level=0).max()

# Unir al dataframe original
df2 = df.join(amenidades_dummies)

In [216]:
detalles_df = df2['detalles'].apply(pd.Series)
df2 = pd.concat([df2, detalles_df], axis=1)
df2.drop(columns=['amenidades', 'detalles'], inplace=True)

In [217]:
df2.sample(6)

,titulo,precio,ubicacion,recamaras,banos,m2,url,fecha,parqueo,1 Studio,...,Número de Piso,Pet friendly,Tipo de propiedad,Niveles de la propiedad,Área total,Área construida,Balcón/Terraza,Piscina,Antigüedad,Tipo de piso
16,Alquiler De Apartamento En San Pedro | 1 Hab |...,297000,"San Pedro, Montes de Oca",NaN,1.0,60.0,https://www.encuentra24.com/costa-rica-es/bien...,14/04/2026,False,1,...,1,NaN,Apartamento,1,60 m²,60 m²,NaN,NaN,NaN,Cerámica
2,"Alquiler De Apartamento De Lujo, Nuevo De 3 Ha...",1300000,"Sabanilla, Montes de Oca",3.0,2.5,156.0,https://www.encuentra24.com/costa-rica-es/bien...,06/04/2026,True,0,...,2,NaN,Apartamento,1,156 m²,156 m²,NaN,NaN,156,NaN
42,Apartamento En Alquiler En Los Yoses. Montes D...,1100,"San Pedro, Montes de Oca",2.0,1.0,NaN,https://www.encuentra24.com/costa-rica-es/bien...,17/04/2026,True,0,...,62 m²,14,Apartamento,NaN,62 m²,NaN,Sí,Apartamento,NaN,NaN
53,Apartamentos En San Pedro *servicios Incluidos*,250000,"San Pedro, Montes de Oca",NaN,1.0,50.0,https://www.encuentra24.com/costa-rica-es/bien...,16/04/2026,True,0,...,1,NaN,Apartamento,3,1000 m²,50 m²,NaN,NaN,4,NaN
55,Alquiler Penthouse De Lujo En Latitud Los Yoses,2600,"San Pedro, Montes de Oca",4.0,3.0,NaN,https://www.encuentra24.com/costa-rica-es/bien...,16/04/2026,True,0,...,22,NaN,Apartamento,NaN,140 m²,NaN,NaN,Sí,2017,Cerámica
22,Alquiler De Amplio Apartamento De Dos Dormitor...,600000,"San Pedro, Montes de Oca",2.0,2.0,NaN,https://www.encuentra24.com/costa-rica-es/bien...,17/04/2026,False,0,...,NaN,NaN,Apartamento,NaN,NaN,NaN,Apartamento,NaN,NaN,NaN


In [218]:
df2.columns

Index(['titulo', 'precio', 'ubicacion', 'recamaras', 'banos', 'm2', 'url',
       'fecha', 'parqueo', '1 Studio', '2 o mas Studio', '2 o más elevadores',
       'Amoblado', 'Baño visitas', 'Calentador de agua', 'Calle sin salida',
       'Cerca de Escuela', 'Cerca del Tráfico', 'Cocina amoblada',
       'Comedor de diario', 'Cuarto y baño de empleada', 'Depósito',
       'Desayunador', 'Despensa', 'Dispensador de agua caliente instantáneo',
       'Elegante lobby', 'Estufa', 'Garaje techado', 'Gimnasio',
       'Jardín o Parque', 'Lavadora', 'Lavandería interna', 'Lavaplatos',
       'Microondas', 'Nevera', 'Parking bajo techo', 'Parking de visitas',
       'Parque Infantil', 'Patio', 'Sala y Comedor', 'Salón de fiestas',
       'Seguridad 24 Horas', 'Servicios', 'Terreno en esquina',
       'Vista a las Montañas', 'Vista al Lago', 'Área Social',
       'Área de barbacoa', 'Número de Piso', 'Pet friendly',
       'Tipo de propiedad', 'Niveles de la propiedad', 'Área total',
       'Áre

In [219]:
df2.shape

(60, 58)

In [221]:
df2.to_csv("alquileres_encuentra24.csv", index=False)